# Data Stats

Computes the summary statistics described in `data/README.md`.

In [1]:
import json
from pathlib import Path
from collections import Counter, defaultdict

DATA_DIR = Path(".")  # run from data/
TRANSCRIPTS = DATA_DIR / "transcripts" / "step_up.jsonl"
ANNOTATIONS = DATA_DIR / "teacher_annotations" / "step_up_annotations.jsonl"

## transcripts/step_up.jsonl

In [2]:
transcripts = []
with open(TRANSCRIPTS) as f:
    for line in f:
        line = line.strip()
        if line:
            transcripts.append(json.loads(line))

print(f"Total records: {len(transcripts):,}")

Total records: 21,445


In [3]:
# Records per batch
batch_counts = Counter(r["batch"] for r in transcripts)
print("Records per batch:")
for batch in sorted(batch_counts):
    print(f"  {batch}: {batch_counts[batch]:,}")

Records per batch:
  2025-10-16: 462
  2026-02-13: 109
  2026-03-24: 10,878
  2026-04-27: 9,996


In [4]:
# Turns per session
turn_counts = [len(r["turns"]) for r in transcripts]
avg_turns = sum(turn_counts) / len(turn_counts)
print(f"Turns per session — avg: {avg_turns:.0f}, min: {min(turn_counts)}, max: {max(turn_counts)}")

Turns per session — avg: 349, min: 5, max: 1789


In [5]:
# has_video distribution
video_counts = Counter(r["has_video"] for r in transcripts)
print(f"has_video=True:  {video_counts[True]:,} ({100*video_counts[True]/len(transcripts):.1f}%)")
print(f"has_video=False: {video_counts[False]:,} ({100*video_counts[False]/len(transcripts):.1f}%)")

has_video=True:  20,983 (97.8%)
has_video=False: 462 (2.2%)


In [6]:
# Session duration distribution (in minutes)
durations = []
for r in transcripts:
    turns = r["turns"]
    if turns:
        duration_sec = turns[-1]["end_seconds"] - turns[0]["start_seconds"]
        durations.append(duration_sec / 60)

avg_dur = sum(durations) / len(durations)
print(f"Session duration (minutes) — avg: {avg_dur:.1f}, min: {min(durations):.1f}, max: {max(durations):.1f}")

Session duration (minutes) — avg: 52.8, min: 0.9, max: 180.4


In [7]:
# Unique tutors and students
tutor_ids = {r["session"]["tutor_id"] for r in transcripts}
student_ids = {r["session"]["student_id"] for r in transcripts}
print(f"Unique tutors:  {len(tutor_ids):,}")
print(f"Unique students: {len(student_ids):,}")

Unique tutors:  1,116
Unique students: 1,302


## teacher_annotations/step_up_annotations.jsonl

In [8]:
annotations = []
with open(ANNOTATIONS) as f:
    for line in f:
        line = line.strip()
        if line:
            annotations.append(json.loads(line))

print(f"Total records: {len(annotations):,}")

Total records: 1,095


In [9]:
# Records per annotation type
type_counts = Counter(r["annotation_type"] for r in annotations)
print("Records per annotation_type:")
for atype in sorted(type_counts):
    print(f"  {atype}: {type_counts[atype]}")

Records per annotation_type:
  caption: 79
  rapport: 705
  scaffolding: 311


In [10]:
# Records per batch
ann_batch_counts = Counter(r["batch"] for r in annotations)
print("Records per batch:")
for batch in sorted(ann_batch_counts):
    print(f"  {batch}: {ann_batch_counts[batch]}")

Records per batch:
  2025-10-16: 1095


In [11]:
# Null turn_number_start/end in turn_annotations, by type
for atype in ("rapport", "scaffolding"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    total_ta = sum(len(r["turn_annotations"]) for r in records)
    null_ta = sum(
        1
        for r in records
        for ta in r["turn_annotations"]
        if ta.get("turn_number_start") is None and ta.get("turn_number_end") is None
    )
    pct = 100 * null_ta / total_ta if total_ta else 0
    print(f"{atype}: {null_ta} / {total_ta} null-anchored turn_annotations ({pct:.1f}%)")

rapport: 23 / 4936 null-anchored turn_annotations (0.5%)
scaffolding: 15 / 1632 null-anchored turn_annotations (0.9%)


In [12]:
# Annotated sessions (unique transcript_ids with scaffolding or rapport)
annotated_ids = {r["transcript_id"] for r in annotations if r["annotation_type"] in ("scaffolding", "rapport")}
print(f"Sessions with scaffolding or rapport annotations: {len(annotated_ids)}")

# Moments per type (excluding null-anchored)
for atype in ("scaffolding", "rapport"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    moments = [
        ta
        for r in records
        for ta in r["turn_annotations"]
        if ta.get("turn_number_start") is not None
    ]
    print(f"{atype} moments (anchored): {len(moments)}")

Sessions with scaffolding or rapport annotations: 209
scaffolding moments (anchored): 1617
rapport moments (anchored): 4913
